# PneumoniaMNIST + MedSymmFlow Synthetic Augmentation

**Does augmenting PneumoniaMNIST with MedSymmFlow-generated images improve a ResNet-18 classifier on the held-out test split?**

This notebook is a *runnable core* of the v2.0 augmentation protocol. It runs top-to-bottom in one Colab session (T4 GPU) and implements the methodologically load-bearing parts:

| Protocol item | Implemented here |
|---|---|
| **G1 split hygiene** — explicit `val` (524) / `test` (624), never select on test | Uses `medmnist` genuine splits directly (not the repo loaders that alias val→test); asserted |
| **G3** — 28 px, **MSF** generator variant | Zenodo MSF `RGB_28` checkpoint |
| **Arms** | **B0** (real only), **S1** (synthetic pretrain -> real fine-tune), **C1** (MSF reference) |
| **Data-scaling sweep** | `BUDGETS` knob, stratified subsampling, fixed seeds |
| **Filtering (Sec 7)** | Memorisation NN screen + confidence filter |
| **Evaluation** | Test **AUC** primary, threshold fixed on val, multi-seed mean +/- std |

The remaining arms (B1, B2, S2, S3, D1), the beta / ODE-step / ratio sweeps, 5-seed CIs and 224 px are left as documented extensions in the final section — the code is structured so they plug in.

> **Run order:** top to bottom. Everything downstream depends on the setup and data cells. There is no "resume from Drive" shortcut at the top — that was the ordering bug in the previous notebook.


## 0. Environment & setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone the fork that already contains the generation-script fixes
# (batch_size + sys.argv isolation), so no runtime patching is needed.
%cd /content
![ -d MedSymmFlow ] || git clone https://github.com/RonNekrashevich/MedSymmFlow.git
%cd /content/MedSymmFlow
!git pull -q

In [ ]:
!pip install -q medmnist torchdiffeq diffusers accelerate zuko scikit-learn

In [ ]:
import os, sys, json, shutil, subprocess, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU"
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
# All paths and knobs live here.
SAVE_DIR = Path("/content/drive/MyDrive/MedSymmFlow_Project")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SYNTHETIC_DIR = SAVE_DIR / "synthetic_28"          # generated PNGs (persisted to Drive)
FILTERED_DIR  = SAVE_DIR / "synthetic_28_filtered" # after memorisation + confidence filtering
RESULTS_PATH  = SAVE_DIR / "results.csv"
for d in (SYNTHETIC_DIR, FILTERED_DIR):
    d.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 28                       # G3: 28 px MSF
CHECKPOINT_PATH = "/content/MedSymmFlow/models_extracted/models/SymmetricalFlowMatchingClass/RGB_28/FM_pneumoniamnist_beta4.0_rgb.pt"

# ---- experiment knobs -------------------------------------------------------
QUICK = True          # True = fast smoke test; set False for the real run
if QUICK:
    BUDGETS = [500]            # real-image training budgets
    SEEDS = [0]               # >= 5 for the final protocol run
    EPOCHS = 5
    SYN_PER_CLASS = 200       # synthetic images per class to generate
else:
    BUDGETS = [500, 4708]
    SEEDS = [0, 1, 2]
    EPOCHS = 15
    SYN_PER_CLASS = 1000

GEN_BETA = 4.0        # paper default for PneumoniaMNIST MSF
GEN_STEP_SIZE = 0.04  # euler, ~25 steps
GEN_CHUNK = 200       # images per class per subprocess call (reduce if OOM)

print("SAVE_DIR:", SAVE_DIR)
print("QUICK:", QUICK, "| BUDGETS:", BUDGETS, "| SEEDS:", SEEDS, "| EPOCHS:", EPOCHS)

## 1. Data & split hygiene (G1)

The PDF flags a trap: MedSymmFlow's *repo* dataloaders alias the validation loader to `split='test'`. We sidestep it entirely by loading the **genuine** `medmnist` splits directly and asserting their sizes. `val` drives all selection; `test` is read only for final reporting.

In [ ]:
from medmnist import PneumoniaMNIST
from torchvision import transforms

# Grayscale radiographs -> 3-channel for an ImageNet-pretrained backbone.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_set = PneumoniaMNIST(split="train", transform=train_tf, download=True, size=IMAGE_SIZE)
train_set_eval = PneumoniaMNIST(split="train", transform=eval_tf, download=True, size=IMAGE_SIZE)  # for filtering
val_set   = PneumoniaMNIST(split="val",   transform=eval_tf, download=True, size=IMAGE_SIZE)
test_set  = PneumoniaMNIST(split="test",  transform=eval_tf, download=True, size=IMAGE_SIZE)

# G1 assertions
assert len(train_set) == 4708, len(train_set)
assert len(val_set) == 524, len(val_set)
assert len(test_set) == 624, len(test_set)
print("Split sizes OK: train 4708 / val 524 / test 624")

In [ ]:
# Per-split class counts and the train/test prevalence shift (Sec 4).
def class_counts(ds):
    y = np.array(ds.labels).reshape(-1)
    return int((y == 0).sum()), int((y == 1).sum())

rows = []
for name, ds in [("train", train_set), ("val", val_set), ("test", test_set)]:
    n0, n1 = class_counts(ds)
    rows.append({"split": name, "normal": n0, "pneumonia": n1, "pneumonia_frac": round(n1/(n0+n1), 3)})
prevalence = pd.DataFrame(rows)
display(prevalence)
print("Prevalence shift: train is more pneumonia-weighted than test -> prefer threshold-free AUC.")

## 2. Classifier & training utilities

ResNet-18 with the stem adapted for 28 px input (3x3 stride-1 conv, no maxpool — standard MedMNIST practice); the ImageNet-pretrained body is kept. `val`-based checkpoint selection and a `val`-fixed decision threshold implement the "never select on test" rule.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score

def build_resnet18(num_classes=2, pretrained=True):
    model = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)  # 28 px stem
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def _loader(ds, batch_size=64, shuffle=False):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    losses, ys, ps = [], [], []
    for images, labels in loader:
        images = images.to(device)
        labels = labels.squeeze().long().to(device)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
        ys.append(labels.detach().cpu().numpy())
        ps.append(torch.softmax(logits, 1)[:, 1].detach().cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return {"loss": float(np.mean(losses)), "auc": roc_auc_score(y, p)}

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    ys, ps = [], []
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        ps.append(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        ys.append(labels.squeeze().long().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def best_threshold_on_val(model):
    y, p = predict_probs(model, _loader(val_set))
    ts = np.linspace(0.05, 0.95, 19)
    j = [balanced_accuracy_score(y, (p >= t).astype(int)) for t in ts]
    return float(ts[int(np.argmax(j))])

def evaluate_on_test(model, threshold):
    y, p = predict_probs(model, _loader(test_set))
    pred = (p >= threshold).astype(int)
    return {
        "test_auc": roc_auc_score(y, p),
        "test_acc": accuracy_score(y, pred),
        "test_balacc": balanced_accuracy_score(y, pred),
        "test_f1": f1_score(y, pred),
    }

def class_weights_for(subset_labels):
    counts = np.bincount(subset_labels, minlength=2)
    w = counts.sum() / (2.0 * np.maximum(counts, 1))
    return torch.tensor(w, dtype=torch.float32, device=device)

def train_classifier(train_ds, train_labels, seed, epochs=EPOCHS, lr=1e-4,
                     init_state=None, weighted=True, tag=""):
    set_seed(seed)
    model = build_resnet18()
    if init_state is not None:
        model.load_state_dict(init_state)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(train_labels) if weighted else None)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = _loader(train_ds, shuffle=True)
    best_auc, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        run_epoch(model, loader, criterion, optimizer)
        vy, vp = predict_probs(model, _loader(val_set))
        val_auc = roc_auc_score(vy, vp)
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"  [{tag} seed {seed}] best val AUC {best_auc:.4f}")
    return model, best_auc

In [ ]:
# Stratified subsampling of the real training set at a fixed seed (Sec 5.1).
train_labels_all = np.array(train_set.labels).reshape(-1)

def stratified_subset(n, seed):
    if n >= len(train_set):
        return list(range(len(train_set))), train_labels_all
    rng = np.random.default_rng(seed)
    idx = []
    for c in (0, 1):
        c_idx = np.where(train_labels_all == c)[0]
        take = int(round(n * (len(c_idx) / len(train_labels_all))))
        idx.extend(rng.choice(c_idx, size=take, replace=False).tolist())
    idx = sorted(idx)
    return idx, train_labels_all[idx]

## 3. Arm B0 — real-only baseline + reproduction check

B0 is the reference every synthetic arm must beat. The full-budget seed-0 model is also reused later as the real-only scorer for the confidence filter (Sec 5).

In [ ]:
results = []  # accumulates one row per (arm, budget, seed)

def add_result(arm, budget, seed, metrics, val_auc):
    row = {"arm": arm, "budget": budget, "seed": seed, "val_auc": val_auc, **metrics}
    results.append(row)
    print(f"  -> {arm} n={budget} seed={seed}: test AUC {metrics['test_auc']:.4f} acc {metrics['test_acc']:.4f}")

b0_models = {}  # (budget, seed) -> trained model, kept for reuse
for budget in BUDGETS:
    for seed in SEEDS:
        idx, sub_labels = stratified_subset(budget, seed)
        sub = Subset(train_set, idx)
        model, val_auc = train_classifier(sub, sub_labels, seed, tag=f"B0 n={budget}")
        thr = best_threshold_on_val(model)
        add_result("B0", budget, seed, evaluate_on_test(model, thr), val_auc)
        b0_models[(budget, seed)] = model

print("\nB0 reproduction target (protocol): ResNet-18 @28px AUC ~= 94.4")

## 4. Synthetic generation — MSF @ 28 px

Downloads the pretrained MedSymmFlow weights (Zenodo) and samples class-conditioned images via the fixed generator script. Generation runs in chunks with distinct seeds so images are diverse and deterministic; everything is written to Drive so a runtime restart doesn't lose it.

In [ ]:
%cd /content/MedSymmFlow
if not os.path.exists("models.zip"):
    !wget -q -O models.zip "https://zenodo.org/records/16086025/files/models.zip?download=1"
!ls -lh models.zip
if not os.path.exists("models_extracted"):
    !unzip -q models.zip -d models_extracted
print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
assert os.path.exists(CHECKPOINT_PATH), CHECKPOINT_PATH

In [ ]:
def generate_synthetic(per_class, chunk=GEN_CHUNK, base_seed=1000, out_dir=SYNTHETIC_DIR):
    # Skip if already generated (persisted on Drive).
    meta_path = out_dir / "metadata.csv"
    if meta_path.exists():
        existing = pd.read_csv(meta_path)
        if (existing["label"] == 0).sum() >= per_class and (existing["label"] == 1).sum() >= per_class:
            print("Synthetic set already present:", len(existing), "images")
            return existing
    for c in ("normal", "pneumonia"):
        (out_dir / c).mkdir(parents=True, exist_ok=True)

    rows, chunk_id = [], 0
    for start in range(0, per_class, chunk):
        n = min(chunk, per_class - start)
        tmp = Path(f"/content/_gen_chunk_{chunk_id}")
        if tmp.exists():
            shutil.rmtree(tmp)
        cmd = [
            "python", "project/generate_pneumoniamnist.py",
            "--checkpoint", CHECKPOINT_PATH,
            "--dataset", "pneumoniamnist", "--n_classes", "2",
            "--num_normal", str(n), "--num_pneumonia", str(n),
            "--seed", str(base_seed + chunk_id),
            "--beta", str(GEN_BETA), "--image_size", str(IMAGE_SIZE),
            "--rgb_mask", "--solver", "euler", "--step_size", str(GEN_STEP_SIZE),
            "--output_dir", str(tmp),
        ]
        env = dict(os.environ, PYTHONPATH="/content/MedSymmFlow/src")
        print("chunk", chunk_id, "->", n, "per class")
        res = subprocess.run(cmd, cwd="/content/MedSymmFlow", env=env,
                             capture_output=True, text=True)
        if res.returncode != 0:
            print(res.stdout[-2000:]); print(res.stderr[-2000:])
            raise RuntimeError("generation failed")
        for cls_name, label in [("normal", 0), ("pneumonia", 1)]:
            for png in sorted((tmp / cls_name).glob("*.png")):
                dst = out_dir / cls_name / f"{cls_name}_{chunk_id:03d}_{png.stem}.png"
                shutil.copy(png, dst)
                rows.append({"image_path": str(dst), "label": label,
                             "class_name": cls_name, "gen_seed": base_seed + chunk_id})
        shutil.rmtree(tmp)
        chunk_id += 1

    meta = pd.DataFrame(rows)
    meta.to_csv(meta_path, index=False)
    print("Generated", len(meta), "synthetic images ->", out_dir)
    return meta

synthetic_meta = generate_synthetic(SYN_PER_CLASS)
display(synthetic_meta.head())

In [ ]:
# Quick visual sanity check (Sec 7.3 visual inspection).
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for r, cls in enumerate(["normal", "pneumonia"]):
    paths = synthetic_meta[synthetic_meta.class_name == cls]["image_path"].tolist()[:8]
    for a, pth in zip(axes[r], paths):
        a.imshow(Image.open(pth), cmap="gray"); a.set_title(cls, fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## 5. Filtering the generated images (Sec 7)

Two mandatory screens before any synthetic image is used for training:

1. **Memorisation screen** — embed real-train and synthetic images with a fixed ImageNet encoder, take each synthetic image's nearest-neighbour distance to the real set, discard the closest ~1.5% (near-copies recycle real data and inflate gains).
2. **Confidence filter** — score synthetic images with the real-only B0 model; keep only confident, label-consistent samples.

In [ ]:
# Fixed feature encoder (unmodified ImageNet ResNet-18, 224 px) for the memorisation screen.
from torchvision.models import resnet18 as tv_resnet18

_enc = tv_resnet18(weights=ResNet18_Weights.DEFAULT)
_enc.fc = nn.Identity()
_enc = _enc.to(device).eval()

_embed_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PathDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = list(paths), list(labels), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("L")), self.labels[i]

@torch.no_grad()
def embed(loader):
    out = []
    for x, _ in loader:
        out.append(nn.functional.normalize(_enc(x.to(device)), dim=1).cpu())
    return torch.cat(out)

# Real train images -> temp PNGs for a uniform embedding path.
real_dir = Path("/content/_real_train_png"); real_dir.mkdir(exist_ok=True)
real_paths = []
raw_train = PneumoniaMNIST(split="train", download=True, size=IMAGE_SIZE)
for i in range(len(raw_train)):
    p = real_dir / f"r_{i:05d}.png"
    if not p.exists():
        raw_train[i][0].convert("L").save(p)
    real_paths.append(str(p))

real_emb = embed(_loader(PathDataset(real_paths, [0]*len(real_paths), _embed_tf), batch_size=128))
syn_emb  = embed(_loader(PathDataset(synthetic_meta.image_path, synthetic_meta.label.tolist(), _embed_tf), batch_size=128))

# Nearest real neighbour distance for each synthetic image (cosine -> distance).
nn_sim = (syn_emb @ real_emb.T).max(dim=1).values.numpy()
nn_dist = 1.0 - nn_sim
cut = np.quantile(nn_dist, 0.015)  # discard closest ~1.5%
keep_mem = nn_dist > cut

plt.hist(nn_dist, bins=40); plt.axvline(cut, color="r", ls="--")
plt.xlabel("nearest-real distance"); plt.ylabel("count"); plt.title("Memorisation screen"); plt.show()
print(f"Memorisation discard: {int((~keep_mem).sum())}/{len(keep_mem)} ({(~keep_mem).mean()*100:.1f}%)")

In [ ]:
# Confidence filter: real-only B0 model (full budget, seed 0) must agree & be confident.
scorer_key = (max(BUDGETS), SEEDS[0])
scorer = b0_models[scorer_key]

syn_ds_eval = PathDataset(synthetic_meta.image_path, synthetic_meta.label.tolist(), eval_tf)
sy, sp = predict_probs(scorer, _loader(syn_ds_eval))
pred_label = (sp >= 0.5).astype(int)
conf = np.where(np.array(synthetic_meta.label) == 1, sp, 1 - sp)
keep_conf = (pred_label == np.array(synthetic_meta.label)) & (conf >= 0.60)

keep = keep_mem & keep_conf
filtered = synthetic_meta[keep].reset_index(drop=True)
filtered.to_csv(FILTERED_DIR / "metadata.csv", index=False)
print(f"Confidence filter keeps {int(keep_conf.sum())}/{len(keep_conf)}")
print(f"Combined kept: {len(filtered)}/{len(synthetic_meta)} "
      f"(normal {int((filtered.label==0).sum())}, pneumonia {int((filtered.label==1).sum())})")

## 6. Arm S1 — synthetic pretrain -> real fine-tune

The best-supported strategy in the protocol: pretrain on filtered synthetic images, then fine-tune all layers on the real budget at 10x lower LR, with `val`-based selection.

In [ ]:
def make_synth_train_ds():
    return PathDataset(filtered.image_path, filtered.label.tolist(), train_tf)

for seed in SEEDS:
    # Pretrain on synthetic once per seed.
    syn_labels = np.array(filtered.label)
    pre_model, _ = train_classifier(make_synth_train_ds(), syn_labels, seed,
                                    epochs=EPOCHS, lr=1e-4, tag="S1-pretrain")
    pre_state = {k: v.detach().cpu().clone() for k, v in pre_model.state_dict().items()}
    for budget in BUDGETS:
        idx, sub_labels = stratified_subset(budget, seed)
        sub = Subset(train_set, idx)
        model, val_auc = train_classifier(sub, sub_labels, seed, epochs=EPOCHS,
                                          lr=1e-5, init_state=pre_state, tag=f"S1 n={budget}")
        thr = best_threshold_on_val(model)
        add_result("S1", budget, seed, evaluate_on_test(model, thr), val_auc)

## 7. Arm C1 — MedSymmFlow reference (distillation control)

**Mandatory and load-bearing.** MSF classifies as well as it generates, and at 28 px it already beats ResNet-18 (AUC 95.2 vs 94.4). So if S1 improves over B0, *distillation of MSF's decision function* is the leading explanation — not "synthetic data adds information."

Reproducing MSF-as-classifier requires the repo's classification path; that's a documented extension. Here we record the published reference so every synthetic arm is reported against it.

In [ ]:
# Published MSF (28 px) test-split reference (protocol Sec 5.3).
C1_AUC, C1_ACC = 0.952, 0.880
for budget in BUDGETS:
    results.append({"arm": "C1", "budget": budget, "seed": -1, "val_auc": np.nan,
                    "test_auc": C1_AUC, "test_acc": C1_ACC,
                    "test_balacc": np.nan, "test_f1": np.nan})
print(f"C1 MSF reference recorded: AUC {C1_AUC}, ACC {C1_ACC}")
print("Extension: reproduce via the repo's classification path on split='test'.")

## 8. Results & comparison

Mean +/- std across seeds per (arm, budget). The decision rule (Sec 9): a synthetic arm is effective at a budget only if it beats the strongest baseline by a pre-registered margin with non-overlapping CIs — **and** the gain is not fully explained by C1.

In [ ]:
res = pd.DataFrame(results)
res.to_csv(RESULTS_PATH, index=False)

summary = (res.groupby(["arm", "budget"])
             .agg(test_auc_mean=("test_auc", "mean"),
                  test_auc_std=("test_auc", "std"),
                  test_acc_mean=("test_acc", "mean"))
             .reset_index())
display(summary)

# B0 vs S1 delta per budget, with C1 alongside.
pivot = summary.pivot_table(index="budget", columns="arm", values="test_auc_mean")
for col in ["B0", "S1", "C1"]:
    if col not in pivot:
        pivot[col] = np.nan
pivot["S1_minus_B0"] = pivot["S1"] - pivot["B0"]
print("\nTest-AUC by budget:")
display(pivot[["B0", "S1", "C1", "S1_minus_B0"]])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for arm, marker in [("B0", "o"), ("S1", "s")]:
    d = summary[summary.arm == arm].sort_values("budget")
    if len(d):
        ax.errorbar(d.budget, d.test_auc_mean, yerr=d.test_auc_std.fillna(0),
                    marker=marker, capsize=3, label=arm)
ax.axhline(C1_AUC, color="gray", ls="--", label="C1 MSF (0.952)")
ax.axhline(0.944, color="green", ls=":", label="B0 target (0.944)")
ax.set_xlabel("real training images"); ax.set_ylabel("test AUC")
ax.set_title("Gain vs data budget"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Saved results ->", RESULTS_PATH)

## 9. Extensions to the full protocol

This core is structured so the rest of the protocol plugs in:

- **Remaining arms.** `B1` (class-weighted loss — already have `class_weights_for`), `B2` (naive oversampling via a `WeightedRandomSampler`), `S2` (concat real+synthetic in one loader), `S3` (synthetic only to equalise prevalence), `D1` (train synthetic, test real — diagnostic). Each is a variant of the existing `train_classifier` call.
- **Full sweep.** Set `QUICK = False`, `BUDGETS = [250, 500, 1000, 2000, 4708]`, `SEEDS = [0,1,2,3,4]`.
- **Generation sweeps.** `beta` in {1,2,4,6}, ODE steps in {10,25,50}, synthetic:real ratio in {0.5,1,2,5}x — select on `val`, fix the generator checkpoint epoch.
- **Statistics.** Paired test across seeds + Benjamini–Hochberg correction; report 95% CIs.
- **C1 reproduction.** Run the repo's MSF classification on `split='test'` at 28 px.
- **224 px / LatMSF.** Reserve for configs that survive at 28 px (the paper notes LatMSF underperforms MSF on PneumoniaMNIST).

**Reporting reminders:** B0 must reproduce ~0.944 AUC before interpreting any synthetic arm; report C1 next to every synthetic result; never select on `test`.
